---
## Section 1: Loops, State, and "Feeding Output Back In"

The biggest conceptual trip-up isn't loop *syntax* — it's the idea of a loop where **each iteration's output becomes the next iteration's input.** That's what makes something feel like a simulation instead of just "doing a thing 10 times."

### `while True` + `break` vs. `for`

Use `for` when you know the number of repetitions ahead of time (e.g. "try 20 evaluation episodes").
Use `while True:` + `break` when you don't know how long something will run and instead want to stop the moment a *condition* becomes true (e.g. "run until the episode ends").

```python
while True:
    # do something
    if some_condition:
        break
```

### State feeding into itself

```python
state = start
while True:
    state = update(state)   # this iteration's output becomes next iteration's input
    if done(state):
        break
```

This is exactly the shape of `state = next_state` you'll see later in the Q-learning loop, and of updating a ball's position/velocity each frame in pygame.


### Live demo

Run this cell together. A "ball" starts at position 0 with velocity 2. Each tick, position updates using its own previous value. We stop the moment it crosses 100.

In [ ]:
position = 0
velocity = 2
steps = 0

while True:
    position += velocity # <-- output feeds back in as next input
    steps += 1

    # stopping condition
    if position > 100:
        break

print(f"Crossed 100 after {steps} steps. Final position: {position}")

### 🧪 Exercise 1

Write a loop that starts a ball at `position = 0` with `velocity = 3`.

- Each tick, update `position`.
- Every 5th step, increase `velocity` by 1 (the ball speeds up — like gravity, sort of).
- Stop the loop once `position > 50`, and print how many steps it took.

Use a `while True:` + `break` — no `for` loop here, since we don't know in advance how many steps it'll take.


In [ ]:
position = 0
velocity = 3
steps = 0

while True:
    position += velocity
    steps += 1

    if steps % 5 == 0:
        velocity += 1

    if position > 50:
        break

print(f"Done after {steps} steps. Final position: {position}, final velocity: {velocity}")

---
## Section 2: Functions, Defaults, and Multi-Value Returns

Two patterns you'll see constantly:

**1. Default arguments** — lets you call a function with or without overriding certain values:
```python
def step(state, velocity=1):
    ...
```

**2. Returning multiple values, then unpacking them** — a function can return several things at once as a tuple, and you can unpack them directly into named variables:
```python
def move(x, y):
    return x + 1, y + 2

new_x, new_y = move(3, 4)   # tuple unpacking — same trick as `a, b = 1, 2`
```

You'll see this exact shape in Gymnasium: `observation, reward, terminated, truncated, info = env.step(action)` — one function call, five values back, all unpacked in one line.

### Combining stop conditions with `or`

```python
done = terminated or truncated   # True if EITHER is True
```


### 🧪 Exercise 2

Refactor the ball simulation into a function.

Write `def step(position, velocity):` that:
- returns the **new position**, and
- a boolean `done` that's `True` once `position > 50`

Then write a loop that calls `step(...)`, unpacks the result, updates `position`, and breaks when `done` is `True`.


In [ ]:
def step(position=0, velocity=0):
    new_position = position + velocity
    done = new_position > 50
    return new_position, done

position = 0
velocity = 3
steps = 0

while True:
    position, done = step(position, velocity)
    steps += 1
    if done:
        break

print(f"Done after {steps} steps. Final position: {position}")

---
## Section 3: Dictionaries as Lookup Tables, Tuples as Keys

This is the pattern most people haven't used much before today, and it's central to how a Q-table works later — so it's worth slowing down for.

### The "look up, or create if missing" pattern

```python
Q = {}                     # empty dictionary: nothing known yet

if state not in Q:
    Q[state] = 0            # create an entry the first time we see this state

Q[state] += 1               # now safe to update it
```

This lets you store information **only for states you've actually seen**, instead of pre-allocating a giant table for every possible state up front (which is often impossible — imagine trying to list every possible sensor reading!).

### Why tuples (not lists) as dictionary keys

Dictionary keys must be **hashable** — essentially, "unchangeable" (immutable). Tuples are immutable (`(2, 1, 4)`), so they work as keys. Lists are mutable (`[2, 1, 4]` can change after creation), so Python won't allow them as keys at all — you'll get an error if you try.

```python
position = (2, 1, 4)        # tuple: OK as a dict key
# position = [2, 1, 4]      # list: NOT allowed as a dict key
```


### Live demo

We'll build a dictionary that counts how many times we visit each `(x, y)` grid coordinate, creating the entry the first time we see it.

In [256]:
import random
visit_counts = {}

for _ in range(20):
    coord = (random.randint(0, 2), random.randint(0, 2)) # a random (x, y) tuple

    if coord not in visit_counts:
        visit_counts[coord] = 0

    visit_counts[coord] += 1

for coord, count in visit_counts.items():
    print(f"{coord}: visited {count} time(s)")

(2, 2): visited 2 time(s)
(2, 0): visited 4 time(s)
(0, 1): visited 2 time(s)
(1, 2): visited 1 time(s)
(0, 0): visited 5 time(s)
(2, 1): visited 3 time(s)
(1, 0): visited 3 time(s)


---
## Section 4: NumPy Essentials

Only the handful of operations you'll actually see later:

| Function | What it does | Where you'll see it |
|---|---|---|
| `np.zeros(n)` | Array of `n` zeros | Initializing Q-values for a new state |
| `np.argmax(array)` | Index of the largest value | "Which action has the best Q-value?" |
| `np.linspace(low, high, n)` | `n` evenly spaced numbers between `low` and `high` | Defining bin edges for discretization |
| `np.digitize(value, edges)` | Which bin a value falls into, given edges | Turning a continuous reading into a bin index |


In [262]:
import numpy as np

# np.zeroes -- a  fresh row of "unknown" values, one per action
q_values = np.zeros(3)
print("q_values", q_values)

# np.argmax -- which index holds the largest value?
q_values = np.array([1.2, 5.7, 3.3])
best_action = np.argmax(q_values)
print("best_action:", best_action)

q_values [0. 0. 0.]
best_action: 1


In [ ]:
# Define 4 bins between -1 and 1 using 3 interior edges
edges = np.linspace(-1, 1, 3)
print("bin edges:", edges)

# Wich bin does each value fall into?
for value in [-0.9, -0.1, 0.4, 0.95]:
    bin_index = np.digitize(value, edges)
    print(f"value {value:>5} -> bin {bin_index}")

bin edges: [-1.  0.  1.]
value  -0.9 -> bin 1
value  -0.1 -> bin 1
value   0.4 -> bin 2
value  0.95 -> bin 2
